# 🎬 DubbingStudioX — Doblaje automático con IA en Google Colab

Evolución de SubtitleStudioX. Ahora con clonación de voz y doblaje a español latino.

---

## 🚀 Uso rápido
1. Ejecuta la **Celda 1** (tarda unos minutos).
2. Ejecuta la **Celda 2** y elige entre procesar desde cero o solo doblaje.
3. Sube tus archivos cuando se te pida y espera el resultado.

In [ ]:
# ═══════════════════════════════════════
# CELDA 1 – ECOSISTEMA SUBTITLESTUDIOX + COQUI-TTS
# ═══════════════════════════════════════
import os
print("✅ Clonando DubbingStudioX...")
!rm -rf DubbingStudioX
!git clone https://github.com/KingEdhard/DubbingStudioX.git
%cd DubbingStudioX

print("✅ Instalando paquetes base (igual que SubtitleStudioX)...")
!pip install faster-whisper==1.2.1 sentencepiece==0.2.1 tqdm==4.67.3 pydub soundfile sacremoses==0.0.4 transformers==5.8.1 torch==2.12.0 --quiet

print("✅ Instalando WhisperX (GPU)...")
!pip install git+https://github.com/m-bain/whisperx.git --quiet

print("✅ Instalando coqui-tts (XTTS‑v2)...")
!pip install coqui-tts --quiet

# FFmpeg
!apt-get install ffmpeg -y > /dev/null
!mkdir -p bin
!ln -sf $(which ffmpeg) bin/ffmpeg.exe
!ln -sf $(which ffprobe) bin/ffprobe.exe

print("✅ Entorno listo (ecosistema SubtitleStudioX + coqui-tts). Ya puedes ejecutar la Celda 2.")

In [ ]:
print("🔍 Verificando módulos...")
try:
    import torch
    print(f"CUDA disponible: {torch.cuda.is_available()}")
    from src.components.doblaje import generar_doblaje
    from src.components.extraccion import extraer_audio_mejorado
    from src.components.traduccion import traducir_srt
    from src.components.muxer import incrustar_subtitulos
    print("✅ Todos los módulos importados correctamente.")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# ═══════════════════════════════════════
# CELDA 2 – MENÚ Y PIPELINE DE DOBLAJE (extrae subs del vídeo)
# ═══════════════════════════════════════
import os, json, shutil, sys, subprocess, re
from google.colab import files
from IPython.display import clear_output
import torch

try:
    from src.components.extraccion import extraer_audio_mejorado
    from src.components.traduccion import traducir_srt
    from src.components.muxer import incrustar_subtitulos
    from src.components.doblaje import generar_doblaje
except Exception as e:
    print(f"❌ Error importando módulos del proyecto: {e}")
    print("Asegúrate de haber ejecutado la Celda 1 correctamente.")
    raise

def transcribir_con_whisperx(wav_path):
    import whisperx
    device, compute_type = "cuda", "float16"
    print("🚀 Iniciando WhisperX Large-v3...")
    model = whisperx.load_model("large-v3", device, compute_type=compute_type, language="en")
    audio = whisperx.load_audio(wav_path)
    result = model.transcribe(audio, batch_size=16)
    print("🎯 Alineación fonética milimétrica (Wav2Vec2)...")
    model_a, metadata = whisperx.load_align_model(language_code="en", device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
    
    srt_path = wav_path.replace("_dialogos_mejorados.wav", "_en.srt")
    def format_time(t):
        h, m, s, ms = int(t // 3600), int((t % 3600) // 60), int(t % 60), int((t % 1) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    with open(srt_path, "w", encoding="utf-8") as f:
        for idx, seg in enumerate(result["segments"], 1):
            f.write(f"{idx}\n{format_time(seg['start'])} --> {format_time(seg['end'])}\n{seg['text'].strip()}\n\n")
    
    json_path = wav_path.replace("_dialogos_mejorados.wav", "_segments.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump([{"start": s["start"], "end": s["end"], "text": s["text"].strip()} for s in result["segments"]], f, ensure_ascii=False, indent=2)
    print(f"✔ Segmentos guardados en {json_path}")
    return srt_path, json_path

def extraer_subtitulos_del_video(video_path):
    """Extrae la pista de subtítulos en español (o inglés) del vídeo.
       Si no hay etiquetas de idioma, usa el segundo flujo de subtítulos (índice 1)."""
    from src.utils import FFPROBE_PATH, FFMPEG_PATH
    # 1. Detectar pistas de subtítulos
    cmd = [FFPROBE_PATH, '-v', 'error', '-select_streams', 's', '-show_entries', 'stream=index:stream_tags=language', '-of', 'json', video_path.replace('\\', '/')]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0 or not res.stdout.strip():
        raise RuntimeError("No se pudo obtener información de subtítulos del vídeo.")
    streams = json.loads(res.stdout).get('streams', [])
    if not streams:
        raise RuntimeError("El vídeo no contiene pistas de subtítulos.")
    
    # Buscar por idioma (español preferido, luego inglés)
    idx_sub = None
    for s in streams:
        lang = (s.get('tags', {}) or {}).get('language', '').lower()
        if lang in ('spa', 'es', 'spanish'):
            idx_sub = s['index']
            break
        elif lang in ('eng', 'en', 'english'):
            if idx_sub is None:
                idx_sub = s['index']
    # Si no se encontró idioma, usar el segundo flujo de subtítulos (índice 1) como fallback
    if idx_sub is None:
        if len(streams) >= 2:
            idx_sub = streams[1]['index']  # segundo flujo
            print("ℹ️ No se detectó idioma, usando el segundo subtítulo.")
        else:
            idx_sub = streams[0]['index']  # único disponible
    
    # 2. Extraer el SRT
    srt_temporal = video_path.rsplit('.',1)[0] + '_extraido.srt'
    cmd_extract = [FFMPEG_PATH, '-y', '-i', video_path.replace('\\', '/'), '-map', f'0:{idx_sub}', srt_temporal]
    subprocess.run(cmd_extract, capture_output=True, text=True)
    if not os.path.exists(srt_temporal):
        raise RuntimeError("Falló la extracción del SRT.")
    
    # 3. Convertir SRT a segmentos
    segments = []
    with open(srt_temporal, 'r', encoding='utf-8') as f:
        content = f.read()
    patron = re.compile(r'\d+\n(\d{2}:\d{2}:\d{2}[.,]\d{3}) --> (\d{2}:\d{2}:\d{2}[.,]\d{3})\n(.+?)(?:\n\n|\Z)', re.DOTALL)
    for start_str, end_str, text in patron.findall(content):
        def to_seconds(tstr):
            tstr = tstr.replace(',', '.')
            h, m, s = map(float, tstr.split(':'))
            return h * 3600 + m * 60 + s
        start = to_seconds(start_str)
        end = to_seconds(end_str)
        text = text.replace('\n', ' ').strip()
        segments.append({"start": start, "end": end, "text": text})
    if not segments:
        raise RuntimeError("El SRT extraído está vacío.")
    json_path = video_path.rsplit('.',1)[0] + '_segments_auto.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(segments, f, ensure_ascii=False, indent=2)
    print(f"✔ Segmentos extraídos: {len(segments)} frases.")
    return json_path

def extraer_audio_del_video(video_path):
    """Extrae la primera pista de audio a WAV mono 22050 Hz."""
    from src.utils import FFMPEG_PATH
    audio_tmp = video_path.rsplit('.',1)[0] + '_audio_ref.wav'
    cmd = [FFMPEG_PATH, '-y', '-i', video_path.replace('\\', '/'), '-ac', '1', '-ar', '22050', audio_tmp]
    subprocess.run(cmd, capture_output=True, text=True)
    if not os.path.exists(audio_tmp):
        raise RuntimeError("Falló la extracción del audio de referencia.")
    return audio_tmp

# ---------- MENÚ PRINCIPAL ----------
print("🎤 DUBBING STUDIO X – Doblaje automático con voz clonada\n")
print("Elige el modo de trabajo:")
print("1. Procesar desde cero (vídeo → doblaje completo)")
print("2. Solo doblaje (el vídeo ya tiene subtítulos en español)")
modo = input("Tu opción (1/2): ").strip()
while modo not in ['1', '2']:
    modo = input("Opción inválida. Elige 1 o 2: ").strip()

try:
    if modo == '1':
        # --- MODO 1: PROCESO COMPLETO ---
        while True:
            try:
                print("\n📤 Sube un archivo de vídeo (mp4, mkv, avi, mov...)")
                uploaded = files.upload()
                if not uploaded:
                    print("No se subió nada. ¿Salir? (S/n)")
                    if input().lower() == 's':
                        break
                    continue
                
                video_nombre = list(uploaded.keys())[0]
                print(f"\n🎬 Procesando: {video_nombre}")
                
                wav = extraer_audio_mejorado(video_nombre)
                if not wav:
                    continue
                
                srt_ingles, json_seg = transcribir_con_whisperx(wav)
                if not srt_ingles or not json_seg:
                    continue
                
                print("\n🌎 Traduciendo a español latino...")
                srt_espanol = traducir_srt(srt_ingles)
                if not srt_espanol:
                    print("⚠ Falló la traducción. Se usará el SRT en inglés como base.")
                    srt_espanol = srt_ingles
                
                print("🔧 Preparando segmentos para doblaje...")
                with open(json_seg, 'r') as f:
                    seg_en = json.load(f)
                from transformers import MarianMTModel, MarianTokenizer
                tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")
                model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-es").to(torch.device('cpu'))
                for seg in seg_en:
                    inputs = tokenizer(seg['text'], return_tensors='pt', padding=True, truncation=True, max_length=512)
                    outputs = model.generate(**inputs)
                    seg['text'] = tokenizer.decode(outputs[0], skip_special_tokens=True)
                json_es = video_nombre.rsplit('.',1)[0] + '_segments_es.json'
                with open(json_es, 'w', encoding='utf-8') as f:
                    json.dump(seg_en, f, ensure_ascii=False, indent=2)
                print(f"✔ Segmentos traducidos guardados en {json_es}")
                
                print("\n🎙️ Generando doblaje con voz clonada (XTTS-v2 GPU)...")
                audio_doblado = generar_doblaje(json_es, wav, "doblaje_temp.wav")
                
                print("\n🎬 Incrustando audio doblado y subtítulos...")
                ruta_final = incrustar_subtitulos(
                    video_nombre,
                    srt_ingles,
                    srt_espanol,
                    formato_salida=None,
                    audio_doblaje=audio_doblado
                )
                if ruta_final:
                    print(f"\n🏁 ¡Vídeo doblado listo!: {ruta_final}")
                    files.download(ruta_final)
                
                for tmp in [wav, audio_doblado]:
                    if os.path.exists(tmp):
                        os.remove(tmp)
            except Exception as e:
                print(f"\n❌ Error procesando el vídeo: {e}")
            finally:
                if input("\n¿Procesar otro vídeo? (S/n): ").lower() == 'n':
                    break
    else:
        # --- MODO 2: SOLO DOBLAJE (extrae todo del vídeo) ---
        while True:
            try:
                print("\n🎥 Sube el vídeo (debe tener subtítulos en español o inglés)")
                uploaded_video = files.upload()
                if not uploaded_video:
                    print("No se subió nada. ¿Salir? (S/n)")
                    if input().lower() == 's':
                        break
                    continue
                video_orig = list(uploaded_video.keys())[0]
                print(f"✅ Vídeo cargado: {video_orig}")
                
                # 1. Extraer segmentos (subs) del vídeo
                print("\n🔍 Extrayendo subtítulos...")
                json_segments = extraer_subtitulos_del_video(video_orig)
                
                # 2. Extraer audio de referencia (primera pista)
                print("\n🎵 Extrayendo audio de referencia...")
                ref_audio = extraer_audio_del_video(video_orig)
                
                print("\n🎙️ Generando doblaje con voz clonada...")
                audio_doblado = generar_doblaje(json_segments, ref_audio, "doblaje_temp.wav")
                
                # 3. Multiplexar (sin añadir nuevos subtítulos, conservamos los originales)
                print("\n🎬 Multiplexando...")
                ruta_final = incrustar_subtitulos(video_orig, None, None, audio_doblaje=audio_doblado)
                if ruta_final:
                    print(f"\n🏁 ¡Vídeo doblado listo!: {ruta_final}")
                    files.download(ruta_final)
                
                # Limpiar temporales
                for tmp in [json_segments, ref_audio, audio_doblado]:
                    if os.path.exists(tmp):
                        os.remove(tmp)
            except Exception as e:
                print(f"\n❌ Error en el proceso de doblaje: {e}")
            finally:
                if input("\n¿Doblar otro? (S/n): ").lower() == 'n':
                    break
except KeyboardInterrupt:
    print("\n⏹️ Proceso interrumpido por el usuario.")
except Exception as e:
    print(f"\n❌ Error inesperado: {e}")
finally:
    print("🎉 ¡Gracias por usar DubbingStudioX!")